## LangGraph Open Deep Research - Supervisor-Researcher Architecture

In this notebook, we'll explore the **supervisor-researcher delegation architecture** for conducting deep research with LangGraph.

You can visit this repository to see the original application: [Open Deep Research](https://github.com/langchain-ai/open_deep_research)

Let's jump in!

## What We're Building

This implementation uses a **hierarchical delegation pattern** where:

1. **User Clarification** - Optionally asks clarifying questions to understand the research scope
2. **Research Brief Generation** - Transforms user messages into a structured research brief
3. **Supervisor** - A lead researcher that analyzes the brief and delegates research tasks
4. **Parallel Researchers** - Multiple sub-agents that conduct focused research simultaneously
5. **Research Compression** - Each researcher synthesizes their findings
6. **Final Report** - All findings are combined into a comprehensive report

![Architecture Diagram](https://private-user-images.githubusercontent.com/181020547/465824799-12a2371b-8be2-4219-9b48-90503eb43c69.png?jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmF3LmdpdGh1YnVzZXJjb250ZW50LmNvbSIsImtleSI6ImtleTUiLCJleHAiOjE3NjAwNDgyMzcsIm5iZiI6MTc2MDA0NzkzNywicGF0aCI6Ii8xODEwMjA1NDcvNDY1ODI0Nzk5LTEyYTIzNzFiLThiZTItNDIxOS05YjQ4LTkwNTAzZWI0M2M2OS5wbmc_WC1BbXotQWxnb3JpdGhtPUFXUzQtSE1BQy1TSEEyNTYmWC1BbXotQ3JlZGVudGlhbD1BS0lBVkNPRFlMU0E1M1BRSzRaQSUyRjIwMjUxMDA5JTJGdXMtZWFzdC0xJTJGczMlMkZhd3M0X3JlcXVlc3QmWC1BbXotRGF0ZT0yMDI1MTAwOVQyMjEyMTdaJlgtQW16LUV4cGlyZXM9MzAwJlgtQW16LVNpZ25hdHVyZT1iYTRmYTAzYjkzYjA2MGE4ZTZlYjQ4ODU1OWIwY2VlZWU0Mzk0YzdmMjQ1YTlhMDMyNmI3NWNlZTQxNDdlZGViJlgtQW16LVNpZ25lZEhlYWRlcnM9aG9zdCJ9.a8477QD1J4Lrmys7jB8gt_H5pdiKBsKsu3npEqZjEpo)

This differs from a section-based approach by allowing dynamic task decomposition based on the research question, rather than predefined sections.

## Dependencies

You'll need API keys for Anthropic (for the LLM) and Tavily (for web search). We'll configure the system to use Anthropic's Claude Sonnet 4 exclusively.

In [1]:
import os
import getpass

os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ")
os.environ["TAVILY_API_KEY"] = getpass.getpass("Enter your Tavily API key: ")

## Task 1: State Definitions

The state structure is hierarchical with three levels:

### Agent State (Top Level)
Contains the overall conversation messages, research brief, accumulated notes, and final report.

### Supervisor State (Middle Level)
Manages the research supervisor's messages, research iterations, and coordinating parallel researchers.

### Researcher State (Bottom Level)
Each individual researcher has their own message history, tool call iterations, and research findings.

We also have structured outputs for tool calling:
- **ConductResearch** - Tool for supervisor to delegate research to a sub-agent
- **ResearchComplete** - Tool to signal research phase is done
- **ClarifyWithUser** - Structured output for asking clarifying questions
- **ResearchQuestion** - Structured output for the research brief

Let's import these from our library: [`open_deep_library/state.py`](open_deep_library/state.py)

In [2]:
# Import state definitions from the library
from open_deep_library.state import (
    # Main workflow states
    AgentState,           # Lines 65-72: Top-level agent state with messages, research_brief, notes, final_report
    AgentInputState,      # Lines 62-63: Input state is just messages
    
    # Supervisor states
    SupervisorState,      # Lines 74-81: Supervisor manages research delegation and iterations
    
    # Researcher states
    ResearcherState,      # Lines 83-90: Individual researcher with messages and tool iterations
    ResearcherOutputState, # Lines 92-96: Output from researcher (compressed research + raw notes)
    
    # Structured outputs for tool calling
    ConductResearch,      # Lines 15-19: Tool for delegating research to sub-agents
    ResearchComplete,     # Lines 21-22: Tool to signal research completion
    ClarifyWithUser,      # Lines 30-41: Structured output for user clarification
    ResearchQuestion,     # Lines 43-48: Structured output for research brief
)

#### ❓ Question 1:

 Explain the interrelationships between the three states.  Why don't we just make a single huge state?

##Answer##

The Deep Research system uses a **hierarchical state architecture** with three distinct levels, each serving a specific purpose in the workflow:

### 1. **AgentState** (Top Level - Main Orchestrator)

**Role**: Acts as the **master state** that maintains the complete conversation context and aggregates all research outputs.

**Contains**:
- User conversation messages
- The research brief (what needs to be researched)
- Accumulated notes from all research activities
- The final synthesized report

### 2. **SupervisorState** (Middle Level - Research Coordinator)

**Role**: Manages the **research delegation and coordination** phase.

**Contains**:
- Supervisor's internal conversation (planning, delegation decisions)
- Research iteration tracking (to prevent infinite loops)
- Accumulated research notes from delegated tasks

### 3. **ResearcherState** (Bottom Level - Individual Research Tasks)

**Role**: Handles **focused research execution** on specific topics.

**Contains**:
- Individual researcher's conversation history
- Tool call iteration tracking
- Specific research topic assignment
- Research findings and raw notes

## Data Flow Between States

The states interact through a **hierarchical data flow**:

1. **AgentState → SupervisorState**: The research brief flows down
2. **SupervisorState → ResearcherState**: Specific research topics are delegated
3. **ResearcherState → SupervisorState**: Research findings flow back up
4. **SupervisorState → AgentState**: Consolidated notes and findings flow back to main state

## Why Not a Single Huge State?

There are several compelling reasons for this multi-state architecture:

### 1. **Separation of Concerns**
Each state handles a distinct responsibility:
- **AgentState**: User interaction and final synthesis
- **SupervisorState**: Research planning and coordination
- **ResearcherState**: Focused research execution

### 2. **Parallel Execution**
The supervisor can spawn **multiple researcher instances** simultaneously, each with their own isolated state. This enables parallel research on different topics without state conflicts.

### 3. **Iteration Control**
Each level has its own iteration limits:
- Supervisor tracks `research_iterations` (how many research cycles)
- Researcher tracks `tool_call_iterations` (how many tool calls per research task)

This prevents infinite loops at different levels of the hierarchy.

### 4. **Memory Management**
Instead of one massive state with all conversation histories mixed together, each level maintains only the messages relevant to its scope:
- User conversations stay in AgentState
- Supervisor planning stays in SupervisorState  
- Individual research conversations stay in ResearcherState

### 5. **Subgraph Isolation**
LangGraph allows each state to be used in **separate subgraphs**:
- Main graph uses AgentState
- Supervisor subgraph uses SupervisorState
- Researcher subgraph uses ResearcherState

This enables clean composition and reusability.

### 6. **State Transitions**
The architecture supports clean **state transformations** between levels. For example, when research is complete, the supervisor consolidates findings and passes them back to the main agent state, filtering out internal planning conversations.

### 7. **Debugging and Monitoring**
With separate states, you can easily inspect what's happening at each level:
- Is the supervisor making good delegation decisions?
- Are individual researchers finding relevant information?
- Is the final synthesis working correctly?

A single huge state would make this much harder to debug and reason about.

This hierarchical design follows the **principle of least privilege** - each component only has access to the state information it needs to perform its specific function, leading to a more maintainable and scalable system.

## Task 2: Utility Functions and Tools

The system uses several key utilities:

### Search Tools
- **tavily_search** - Async web search with automatic summarization to stay within token limits
- Supports Anthropic native web search and Tavily API

### Reflection Tools
- **think_tool** - Allows researchers to reflect on their progress and plan next steps (ReAct pattern)

### Helper Utilities
- **get_all_tools** - Assembles the complete toolkit (search + MCP + reflection)
- **get_today_str** - Provides current date context for research
- Token limit handling utilities for graceful degradation

These are defined in [`open_deep_library/utils.py`](open_deep_library/utils.py)

In [3]:
# Import utility functions and tools from the library
from open_deep_library.utils import (
    # Search tool - Lines 43-136: Tavily search with automatic summarization
    tavily_search,
    
    # Reflection tool - Lines 219-244: Strategic thinking tool for ReAct pattern
    think_tool,
    
    # Tool assembly - Lines 569-597: Get all configured tools
    get_all_tools,
    
    # Date utility - Lines 872-879: Get formatted current date
    get_today_str,
    
    # Supporting utilities for error handling
    get_api_key_for_model,          # Lines 892-914: Get API keys from config or env
    is_token_limit_exceeded,         # Lines 665-701: Detect token limit errors
    get_model_token_limit,           # Lines 831-846: Look up model's token limit
    remove_up_to_last_ai_message,    # Lines 848-866: Truncate messages for retry
    anthropic_websearch_called,      # Lines 607-637: Detect Anthropic native search usage
    openai_websearch_called,         # Lines 639-658: Detect OpenAI native search usage
    get_notes_from_tool_calls,       # Lines 599-601: Extract notes from tool messages
)

### ❓ Question 2:  

What are the advantages and disadvantages of importing these components instead of including them in the notebook?


## Advantages of Importing Components from `open_deep_library`

### 1. **Modularity and Organization**
- **Separation of Concerns**: Each module has a clear responsibility:
  - `utils.py` (926 lines): Contains utility functions, tool implementations, and helper methods
  - `deep_researcher.py` (719 lines): Contains the main LangGraph workflow logic
  - `state.py`: Defines data structures and state management
  - `configuration.py`: Handles configuration management
  - `prompts.py`: Contains prompt templates

- **Maintainability**: Changes to core logic can be made in one place and automatically propagate to all notebooks using the library.

### 2. **Code Reusability**
- The same components can be used across multiple notebooks or projects without duplication
- Promotes DRY (Don't Repeat Yourself) principles
- Enables building a consistent toolkit for research applications

### 3. **Readability and Focus**
- The notebook remains focused on the educational content and demonstrations
- Complex implementation details are abstracted away, allowing learners to focus on concepts
- The notebook becomes more readable with clear imports showing what functionality is being used

### 4. **Version Control and Collaboration**
- Library code can be version controlled separately from notebooks
- Multiple team members can work on different components simultaneously
- Easier to track changes to core functionality vs. experimental notebook code

### 5. **Testing and Quality Assurance**
- Library components can have comprehensive unit tests
- Code can be linted and formatted consistently
- Reduces the risk of copy-paste errors across notebooks

### 6. **Performance Benefits**
- Code is compiled once when imported rather than re-executed in each cell
- Shared imports across multiple notebooks are more memory efficient
- Faster notebook execution since complex logic isn't re-parsed

## Disadvantages of Importing Components

### 1. **Reduced Self-Containment**
- The notebook is no longer completely self-contained
- Dependencies on external files make it harder to share as a standalone document
- Requires the entire library structure to be present for the notebook to work

### 2. **Debugging Complexity**
- Harder to debug issues since the code execution spans multiple files
- Stack traces may point to library files rather than notebook cells
- Students can't easily modify core functionality to experiment

### 3. **Learning Curve Impact**
- Students can't see the full implementation details directly in the notebook
- May create a "black box" effect where the internal workings are hidden
- Requires additional navigation to understand how components work

### 4. **Development Workflow Changes**
- Changes to library code require restarting the kernel to take effect
- No hot-reloading of library changes during development
- More complex setup process for new users

### 5. **Dependency Management**
- Need to ensure library is properly installed and in the Python path
- Version compatibility issues between notebook and library
- Additional complexity in deployment and distribution

## Task 3: Configuration System

The configuration system controls:

### Research Behavior
- **allow_clarification** - Whether to ask clarifying questions before research
- **max_concurrent_research_units** - How many parallel researchers can run (default: 5)
- **max_researcher_iterations** - How many times supervisor can delegate research (default: 6)
- **max_react_tool_calls** - Tool call limit per researcher (default: 10)

### Model Configuration
- **research_model** - Model for research and supervision (we'll use Anthropic)
- **compression_model** - Model for synthesizing findings
- **final_report_model** - Model for writing the final report
- **summarization_model** - Model for summarizing web search results

### Search Configuration
- **search_api** - Which search API to use (ANTHROPIC, TAVILY, or NONE)
- **max_content_length** - Character limit before summarization

Defined in [`open_deep_library/configuration.py`](open_deep_library/configuration.py)

In [4]:
# Import configuration from the library
from open_deep_library.configuration import (
    Configuration,    # Lines 38-247: Main configuration class with all settings
    SearchAPI,        # Lines 11-17: Enum for search API options (ANTHROPIC, TAVILY, NONE)
)

## Task 4: Prompt Templates

The system uses carefully engineered prompts for each phase:

### Phase 1: Clarification
**clarify_with_user_instructions** - Analyzes if the research scope is clear or needs clarification

### Phase 2: Research Brief
**transform_messages_into_research_topic_prompt** - Converts user messages into a detailed research brief

### Phase 3: Supervisor
**lead_researcher_prompt** - System prompt for the supervisor that manages delegation strategy

### Phase 4: Researcher
**research_system_prompt** - System prompt for individual researchers conducting focused research

### Phase 5: Compression
**compress_research_system_prompt** - Prompt for synthesizing research findings without losing information

### Phase 6: Final Report
**final_report_generation_prompt** - Comprehensive prompt for writing the final report

All prompts are defined in [`open_deep_library/prompts.py`](open_deep_library/prompts.py)

In [8]:
# Import prompt templates from the library
from open_deep_library.prompts import (
    clarify_with_user_instructions,                    # Lines 3-41: Ask clarifying questions
    transform_messages_into_research_topic_prompt,     # Lines 44-77: Generate research brief
    lead_researcher_prompt,                            # Lines 79-136: Supervisor system prompt
    research_system_prompt,                            # Lines 138-183: Researcher system prompt
    compress_research_system_prompt,                   # Lines 186-222: Research compression prompt
    final_report_generation_prompt,                    # Lines 228-308: Final report generation
)

## Task 5: Node Functions - The Building Blocks

Now let's look at the node functions that make up our graph. We'll import them from the library and understand what each does.

### The Complete Research Workflow

The workflow consists of 8 key nodes organized into 3 subgraphs:

1. **Main Graph Nodes:**
   - `clarify_with_user` - Entry point that checks if clarification is needed
   - `write_research_brief` - Transforms user input into structured research brief
   - `final_report_generation` - Synthesizes all research into final report

2. **Supervisor Subgraph Nodes:**
   - `supervisor` - Lead researcher that plans and delegates
   - `supervisor_tools` - Executes supervisor's tool calls (delegation, reflection)

3. **Researcher Subgraph Nodes:**
   - `researcher` - Individual researcher conducting focused research
   - `researcher_tools` - Executes researcher's tool calls (search, reflection)
   - `compress_research` - Synthesizes researcher's findings

All nodes are defined in [`open_deep_library/deep_researcher.py`](open_deep_library/deep_researcher.py)

### Node 1: clarify_with_user

**Purpose:** Analyzes user messages and asks clarifying questions if the research scope is unclear.

**Key Steps:**
1. Check if clarification is enabled in configuration
2. Use structured output to analyze if clarification is needed
3. If needed, end with a clarifying question for the user
4. If not needed, proceed to research brief with verification message

**Implementation:** [`open_deep_library/deep_researcher.py` lines 60-115](open_deep_library/deep_researcher.py#L60-L115)

In [9]:
# Import the clarify_with_user node
from open_deep_library.deep_researcher import clarify_with_user

### Node 2: write_research_brief

**Purpose:** Transforms user messages into a structured research brief for the supervisor.

**Key Steps:**
1. Use structured output to generate detailed research brief from messages
2. Initialize supervisor with system prompt and research brief
3. Set up supervisor messages with proper context

**Why this matters:** A well-structured research brief helps the supervisor make better delegation decisions.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 118-175](open_deep_library/deep_researcher.py#L118-L175)

In [10]:
# Import the write_research_brief node
from open_deep_library.deep_researcher import write_research_brief

### Node 3: supervisor

**Purpose:** Lead research supervisor that plans research strategy and delegates to sub-researchers.

**Key Steps:**
1. Configure model with three tools:
   - `ConductResearch` - Delegate research to a sub-agent
   - `ResearchComplete` - Signal that research is done
   - `think_tool` - Strategic reflection before decisions
2. Generate response based on current context
3. Increment research iteration count
4. Proceed to tool execution

**Decision Making:** The supervisor uses `think_tool` to reflect before delegating research, ensuring thoughtful decomposition of the research question.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 178-223](open_deep_library/deep_researcher.py#L178-L223)

In [11]:
# Import the supervisor node (from supervisor subgraph)
from open_deep_library.deep_researcher import supervisor

### Node 4: supervisor_tools

**Purpose:** Executes the supervisor's tool calls, including strategic thinking and research delegation.

**Key Steps:**
1. Check exit conditions:
   - Exceeded maximum iterations
   - No tool calls made
   - `ResearchComplete` called
2. Process `think_tool` calls for strategic reflection
3. Execute `ConductResearch` calls in parallel:
   - Spawn researcher subgraphs for each delegation
   - Limit to `max_concurrent_research_units` (default: 5)
   - Gather all results asynchronously
4. Aggregate findings and return to supervisor

**Parallel Execution:** This is where the magic happens - multiple researchers work simultaneously on different aspects of the research question.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 225-349](open_deep_library/deep_researcher.py#L225-L349)

In [12]:
# Import the supervisor_tools node
from open_deep_library.deep_researcher import supervisor_tools

### Node 5: researcher

**Purpose:** Individual researcher that conducts focused research on a specific topic.

**Key Steps:**
1. Load all available tools (search, MCP, reflection)
2. Configure model with tools and researcher system prompt
3. Generate response with tool calls
4. Increment tool call iteration count

**ReAct Pattern:** Researchers use `think_tool` to reflect after each search, deciding whether to continue or provide their answer.

**Available Tools:**
- Search tools (Tavily or Anthropic native search)
- `think_tool` for strategic reflection
- `ResearchComplete` to signal completion
- MCP tools (if configured)

**Implementation:** [`open_deep_library/deep_researcher.py` lines 365-424](open_deep_library/deep_researcher.py#L365-L424)

In [13]:
# Import the researcher node (from researcher subgraph)
from open_deep_library.deep_researcher import researcher

### Node 6: researcher_tools

**Purpose:** Executes the researcher's tool calls, including searches and strategic reflection.

**Key Steps:**
1. Check early exit conditions (no tool calls, native search used)
2. Execute all tool calls in parallel:
   - Search tools fetch and summarize web content
   - `think_tool` records strategic reflections
   - MCP tools execute external integrations
3. Check late exit conditions:
   - Exceeded `max_react_tool_calls` (default: 10)
   - `ResearchComplete` called
4. Continue research loop or proceed to compression

**Error Handling:** Safely handles tool execution errors and continues with available results.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 435-509](open_deep_library/deep_researcher.py#L435-L509)

In [14]:
# Import the researcher_tools node
from open_deep_library.deep_researcher import researcher_tools

### Node 7: compress_research

**Purpose:** Compresses and synthesizes research findings into a concise, structured summary.

**Key Steps:**
1. Configure compression model
2. Add compression instruction to messages
3. Attempt compression with retry logic:
   - If token limit exceeded, remove older messages
   - Retry up to 3 times
4. Extract raw notes from tool and AI messages
5. Return compressed research and raw notes

**Why Compression?** Researchers may accumulate lots of tool outputs and reflections. Compression ensures:
- All important information is preserved
- Redundant information is deduplicated
- Content stays within token limits for the final report

**Token Limit Handling:** Gracefully handles token limit errors by progressively truncating messages.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 511-585](open_deep_library/deep_researcher.py#L511-L585)

In [15]:
# Import the compress_research node
from open_deep_library.deep_researcher import compress_research

### Node 8: final_report_generation

**Purpose:** Generates the final comprehensive research report from all collected findings.

**Key Steps:**
1. Extract all notes from completed research
2. Configure final report model
3. Attempt report generation with retry logic:
   - If token limit exceeded, truncate findings by 10%
   - Retry up to 3 times
4. Return final report or error message

**Token Limit Strategy:**
- First retry: Use model's token limit × 4 as character limit
- Subsequent retries: Reduce by 10% each time
- Graceful degradation with helpful error messages

**Report Quality:** The prompt guides the model to create well-structured reports with:
- Proper headings and sections
- Inline citations
- Comprehensive coverage of all findings
- Sources section at the end

**Implementation:** [`open_deep_library/deep_researcher.py` lines 607-697](open_deep_library/deep_researcher.py#L607-L697)

In [16]:
# Import the final_report_generation node
from open_deep_library.deep_researcher import final_report_generation

## Task 6: Graph Construction - Putting It All Together

The system is organized into three interconnected graphs:

### 1. Researcher Subgraph (Bottom Level)
Handles individual focused research on a specific topic:
```
START → researcher → researcher_tools → compress_research → END
               ↑            ↓
               └────────────┘ (loops until max iterations or ResearchComplete)
```

### 2. Supervisor Subgraph (Middle Level)
Manages research delegation and coordination:
```
START → supervisor → supervisor_tools → END
            ↑              ↓
            └──────────────┘ (loops until max iterations or ResearchComplete)
            
supervisor_tools spawns multiple researcher_subgraphs in parallel
```

### 3. Main Deep Researcher Graph (Top Level)
Orchestrates the complete research workflow:
```
START → clarify_with_user → write_research_brief → research_supervisor → final_report_generation → END
                 ↓                                       (supervisor_subgraph)
               (may end early if clarification needed)
```

Let's import the compiled graphs from the library.

In [17]:
# Import the pre-compiled graphs from the library
from open_deep_library.deep_researcher import (
    # Bottom level: Individual researcher workflow
    researcher_subgraph,    # Lines 588-605: researcher → researcher_tools → compress_research
    
    # Middle level: Supervisor coordination
    supervisor_subgraph,    # Lines 351-363: supervisor → supervisor_tools (spawns researchers)
    
    # Top level: Complete research workflow
    deep_researcher,        # Lines 699-719: Main graph with all phases
)

## Why This Architecture?

### Advantages of Supervisor-Researcher Delegation

1. **Dynamic Task Decomposition**
   - Unlike section-based approaches with predefined structure, the supervisor can break down research based on the actual question
   - Adapts to different types of research (comparisons, lists, deep dives, etc.)

2. **Parallel Execution**
   - Multiple researchers work simultaneously on different aspects
   - Much faster than sequential section processing
   - Configurable parallelism (1-20 concurrent researchers)

3. **ReAct Pattern for Quality**
   - Researchers use `think_tool` to reflect after each search
   - Prevents excessive searching and improves search quality
   - Natural stopping conditions based on information sufficiency

4. **Flexible Tool Integration**
   - Easy to add MCP tools for specialized research
   - Supports multiple search APIs (Anthropic, Tavily)
   - Each researcher can use different tool combinations

5. **Graceful Token Limit Handling**
   - Compression prevents token overflow
   - Progressive truncation in final report generation
   - Research can scale to arbitrary depths

### Trade-offs

- **Complexity:** More moving parts than section-based approach
- **Cost:** Parallel researchers use more tokens (but faster)
- **Unpredictability:** Research structure emerges dynamically

## Task 7: Running the Deep Researcher

Now let's see the system in action! We'll use it to analyze a PDF document about how people use AI.

### Setup

We need to:
1. Load the PDF document
2. Configure the execution with Anthropic settings
3. Run the research workflow

In [18]:
# Load the PDF document
from pathlib import Path
import PyPDF2

def load_pdf(pdf_path: str) -> str:
    """Load and extract text from PDF."""
    pdf_text = ""
    with open(pdf_path, 'rb') as file:
        pdf_reader = PyPDF2.PdfReader(file)
        for page in pdf_reader.pages:
            pdf_text += page.extract_text() + "\n\n"
    return pdf_text

# Load the PDF about how people use AI
pdf_path = "data/howpeopleuseai.pdf"
pdf_content = load_pdf(pdf_path)

print(f"Loaded PDF with {len(pdf_content)} characters")
print(f"First 500 characters:\n{pdf_content[:500]}...")

Loaded PDF with 112460 characters
First 500 characters:
NBER WORKING PAPER SERIES
HOW PEOPLE USE CHATGPT
Aaron Chatterji
Thomas Cunningham
David J. Deming
Zoe Hitzig
Christopher Ong
Carl Yan Shan
Kevin Wadman
Working Paper 34255
http://www.nber.org/papers/w34255
NATIONAL BUREAU OF ECONOMIC RESEARCH
1050 Massachusetts Avenue
Cambridge, MA 02138
September 2025
We acknowledge help and comments from Joshua Achiam, Hemanth Asirvatham, Ryan 
Beiermeister,  Rachel Brown, Cassandra Duchan Solis, Jason Kwon, Elliott Mokski, Kevin Rao, 
Harrison Satcher,  Gawe...


In [19]:
# Set up the graph with Anthropic configuration
from IPython.display import Markdown, display
import uuid

# Note: deep_researcher is already compiled from the library
# For this demo, we'll use it directly without additional checkpointing
graph = deep_researcher

print("✓ Graph ready for execution")
print("  (Note: The graph is pre-compiled from the library)")

✓ Graph ready for execution
  (Note: The graph is pre-compiled from the library)


### Configuration for Anthropic

We'll configure the system to use:
- **Claude Sonnet 4** for all research, supervision, and report generation
- **Tavily** for web search (you can also use Anthropic's native search)
- **Moderate parallelism** (3 concurrent researchers)
- **Clarification enabled** (will ask if research scope is unclear)

In [26]:
# Configure for Anthropic with moderate settings
config = {
    "configurable": {
        # Model configuration - using Claude Sonnet 4 for everything
        "research_model": "anthropic:claude-sonnet-4-20250514",
        "research_model_max_tokens": 10000,
        
        "compression_model": "anthropic:claude-sonnet-4-20250514",
        "compression_model_max_tokens": 8192,
        
        "final_report_model": "anthropic:claude-sonnet-4-20250514",
        "final_report_model_max_tokens": 10000,
        
        "summarization_model": "anthropic:claude-sonnet-4-20250514",
        "summarization_model_max_tokens": 8192,
        
        # Research behavior
        "allow_clarification": True,
        "max_concurrent_research_units": 1,  # 1 parallel researchers
        "max_researcher_iterations": 2,      # Supervisor can delegate up to 2 times
        "max_react_tool_calls": 3,           # Each researcher can make up to 3 tool calls
        
        # Search configuration
        "search_api": "tavily",  # Using Tavily for web search
        "max_content_length": 50000,
        
        # Thread ID for this conversation
        "thread_id": str(uuid.uuid4())
    }
}

print("✓ Configuration ready")
print(f"  - Research Model: Claude Sonnet 4")
print(f"  - Max Concurrent Researchers: 3")
print(f"  - Max Iterations: 4")
print(f"  - Search API: Tavily")

✓ Configuration ready
  - Research Model: Claude Sonnet 4
  - Max Concurrent Researchers: 3
  - Max Iterations: 4
  - Search API: Tavily


### Execute the Research

Now let's run the research! We'll ask the system to analyze the PDF and provide insights about how people use AI.

The workflow will:
1. **Clarify** - Check if the request is clear (may skip if obvious)
2. **Research Brief** - Transform our request into a structured brief
3. **Supervisor** - Plan research strategy and delegate to researchers
4. **Parallel Research** - Multiple researchers gather information simultaneously
5. **Compression** - Each researcher synthesizes their findings
6. **Final Report** - All findings combined into comprehensive report

In [27]:
# Create our research request with PDF context
research_request = f"""
I have a PDF document about how people use AI. Please analyze this document and provide insights about:

1. What are the main findings about how people are using AI?
2. What are the most common use cases?
3. What trends or patterns emerge from the data?

Here's the PDF content:

{pdf_content[:10000]}  # First 10k chars to stay within limits

...[content truncated for context window]
"""

# Execute the graph
async def run_research():
    """Run the research workflow and display results."""
    print("Starting research workflow...\n")
    
    async for event in graph.astream(
        {"messages": [{"role": "user", "content": research_request}]},
        config,
        stream_mode="updates"
    ):
        # Display each step
        for node_name, node_output in event.items():
            print(f"\n{'='*60}")
            print(f"Node: {node_name}")
            print(f"{'='*60}")
            
            if node_name == "clarify_with_user":
                if "messages" in node_output:
                    last_msg = node_output["messages"][-1]
                    print(f"\n{last_msg.content}")
            
            elif node_name == "write_research_brief":
                if "research_brief" in node_output:
                    print(f"\nResearch Brief Generated:")
                    print(f"{node_output['research_brief'][:500]}...")
            
            elif node_name == "supervisor":
                print(f"\nSupervisor planning research strategy...")
                if "supervisor_messages" in node_output:
                    last_msg = node_output["supervisor_messages"][-1]
                    if hasattr(last_msg, 'tool_calls') and last_msg.tool_calls:
                        print(f"Tool calls: {len(last_msg.tool_calls)}")
                        for tc in last_msg.tool_calls:
                            print(f"  - {tc['name']}")
            
            elif node_name == "supervisor_tools":
                print(f"\nExecuting supervisor's tool calls...")
                if "notes" in node_output:
                    print(f"Research notes collected: {len(node_output['notes'])}")
            
            elif node_name == "final_report_generation":
                if "final_report" in node_output:
                    print(f"\n" + "="*60)
                    print("FINAL REPORT GENERATED")
                    print("="*60 + "\n")
                    display(Markdown(node_output["final_report"]))
    
    print("\n" + "="*60)
    print("Research workflow completed!")
    print("="*60)

# Run the research
await run_research()

Starting research workflow...


Node: clarify_with_user

I have sufficient information to proceed with your analysis. You've provided a comprehensive PDF document titled "How People Use ChatGPT" (NBER Working Paper No. 34255) and requested specific insights about: 1) main findings about how people use AI, 2) most common use cases, and 3) trends/patterns from the data. The document contains detailed research on ChatGPT usage patterns from May 2024 to June 2025, including usage classifications, work vs. non-work patterns, and demographic insights. I will now begin analyzing this research paper to provide you with the requested insights.

Node: write_research_brief

Research Brief Generated:
I have provided a comprehensive NBER working paper titled "How People Use ChatGPT" (Working Paper No. 34255, September 2025) by Chatterji et al. that analyzes ChatGPT usage patterns from May 2024 to June 2025. I need you to thoroughly analyze this specific document and provide detailed insights on thr


Node: research_supervisor

Node: final_report_generation

FINAL REPORT GENERATED



# Comprehensive Analysis of ChatGPT Usage Patterns: Key Findings from NBER Working Paper 34255

This comprehensive analysis examines the groundbreaking NBER working paper "How People Use ChatGPT" by Chatterji et al., which provides the first large-scale empirical study of how consumers actually use LLM chatbots. The research analyzes usage patterns from May 2024 to June 2025, offering unprecedented insights into the adoption, usage evolution, and economic implications of AI chatbot technology.

## Main Findings About AI Usage Patterns

### Unprecedented Growth and Global Adoption

ChatGPT has achieved remarkable adoption rates that surpass any previous technology in history. Launched in November 2022, the platform reached 1 million users within just 5 days and has grown to serve around 10% of the world's adult population by July 2025, representing more than 700 million weekly active users [1][2]. By September 2025, this figure had grown to over 750 million weekly active users [2].

The scale of usage is staggering: by June 2025, users were sending more than 2.6 billion messages per day, or over 30,000 messages per second [2][3]. Weekly message volume reached 18 billion by July 2025 [1]. To put this growth in perspective, ChatGPT reached 1 billion daily messages in December 2024, less than two years after its release, compared to Google Search which took eight years to reach 1 billion daily searches [2].

Weekly active users doubled every 7-8 months since launch, with message volume growing even faster - increasing 5.8x in the last year while user volume grew 3.2x [2]. If current growth trajectories continue, ChatGPT message volume would equal the number of current Google searches in just over a year [2].

### Dramatic Demographic Evolution

The demographic profile of ChatGPT users has undergone significant transformation since launch. Initially, early adopters were overwhelmingly male, with over 80% of users having typically masculine first names in the initial months [1][2]. However, this gender gap has narrowed dramatically and may have closed completely. By July 2025, 52% of active users had typically female names, suggesting the platform now has roughly balanced usage that is slightly female-leaning [2][3].

Age demographics reveal that nearly half of all messages come from users under 26 years old [1][3]. This younger user base represents a significant portion of the platform's engagement, indicating strong adoption among digital natives.

Geographically, the research reveals higher growth rates in lower-income countries compared to wealthier nations [1]. Middle-income countries experienced 5-6x growth compared to 3x growth in the richest countries, with countries like Brazil, South Korea, and the US now having similar usage rates despite vastly different GDP per capita levels [2]. This pattern contradicts typical technology adoption patterns that often exacerbate inequality.

### User Engagement Intensification

User engagement has intensified significantly over time, with all cohorts of users sending substantially more messages per day beginning in early 2025. This suggests significant improvements in ChatGPT's capabilities and user-friendliness [2]. Early adopters from Q1 2023 were sending 40% more messages per day in July 2025 compared to two years earlier [2][3]. Usage patterns showed consistency across all signup cohorts - remaining flat through most of 2024 but increasing substantially beginning in late 2024/early 2025 [2].

## Most Common Use Cases and Classification Taxonomy

### Primary Use Categories

The research employs a sophisticated automated classification system to categorize ChatGPT conversations into distinct taxonomies. The analysis reveals that nearly 80% of all ChatGPT usage falls into three broad categories [1][3]:

**Practical Guidance (29% of usage)** represents the most common use case, encompassing tutoring and teaching, how-to advice about various topics, creative ideation, and health/fitness/beauty/self-care activities [1][3]. Within this category, tutoring and teaching alone accounts for 10.2% of all user messages and 36% of Practical Guidance messages [2]. This translates to more than 260 million learning-related messages sent daily [4], making education a primary driver of ChatGPT adoption [4].

**Seeking Information (24% of usage)** includes searches for specific information, purchasable products, and cooking recipes [1][3]. This category appears to serve as a very close substitute for traditional web search engines, but with the added capability of providing conversational, contextual responses rather than simple link listings.

**Writing (24% of usage)** encompasses the automated production of emails, documents, and other communications, as well as editing, critiquing, summarizing, and translating text provided by users [1][3]. Notably, about two-thirds of all Writing messages ask ChatGPT to modify existing user text (editing, critiquing, translating) rather than creating entirely new content from scratch [2]. The most frequent sub-topic within Writing is "Edit or Critique Provided Text" at 10.6% of all messages [4].

### Work vs. Non-Work Usage Patterns

A striking finding from the research is the dramatic shift in work-related versus non-work-related usage over time. Non-work messages have grown from 53% in mid-2024 to over 70% by mid-2025 [1][3]. By June 2025, non-work-related messages represented 73% of all usage [1].

This shift challenges the conventional wisdom that AI chatbots primarily serve as productivity tools for professional work. Instead, the data suggests that ChatGPT's primary value may lie in supporting everyday decision-making, learning, and creativity outside of formal work contexts [3].

Work usage remains concentrated among educated users in highly-paid professional occupations [1]. When work-related usage does occur, Writing dominates these interactions, accounting for approximately 40% of work-related messages in June 2025 [2]. This highlights chatbots' unique ability to generate digital outputs compared to traditional search engines [1].

### Interaction Types and User Intentions

The research classifies user interactions into three distinct types based on user intentions:

**Asking (~49%)** involves seeking information or clarification to inform a decision [2]. These interactions consistently receive the highest quality ratings both from automated classifiers measuring user satisfaction and from direct user feedback [2].

**Doing (~40%)** encompasses wanting to produce some output or perform a particular task [2]. In work contexts, Doing activities comprise approximately 56% of interactions and focus largely on writing tasks [1][2]. Nearly 35% of all work-related queries are Doing messages related to Writing [2].

**Expressing (~11%)** involves expressing views or feelings without seeking specific information or action [2]. This category has grown significantly over the study period, increasing from just under 8% in July 2024 to 13.8% by June 2025 [2].

### Specialized Use Cases

Contrary to widespread assumptions about AI chatbot usage, computer programming represents only 4.2% of all messages [1][3]. Technical Help, which includes Computer Programming (4.2%), Mathematical Calculations (3%), and Data Analysis (0.4%), comprises a relatively small portion of overall usage [2]. This contrasts sharply with other AI platforms like Claude, where 33% of work-related conversations are programming-related [2].

Self-expression and companionship represent even smaller usage shares, with only 2.4% of all ChatGPT messages related to Relationships and Personal Reflection (1.9%) or Games and Role Play (0.4%) [2]. This finding challenges narratives about AI chatbots primarily serving as social companions.

Multimedia usage has grown from 2% to just over 7% of all interactions, with a significant spike in April 2025 following ChatGPT's release of new image-generation capabilities [2].

## Trends and Patterns in Usage Data

### Evolution of Usage Categories Over Time

The research reveals significant shifts in how people use ChatGPT over the study period. Writing usage has declined from 36% of all usage in July 2024 to 24% a year later, while Seeking Information has grown from 14% to 24% over the same period [2]. Technical Help has declined from 12% of all usage in July 2024 to around 5% a year later [2].

These shifts suggest that as ChatGPT's user base has expanded beyond early adopters (who were more likely to be technically oriented professionals), usage has shifted toward more general-purpose applications like information seeking and practical guidance.

### Behavioral Changes Within User Cohorts

A crucial finding is that the decrease in work-related message share is primarily due to changing usage patterns within each cohort of users rather than changes in the composition of new ChatGPT users [2]. This suggests that individual users are increasingly finding value in ChatGPT for non-work activities over time, rather than the shift being driven solely by different types of people joining the platform.

Asking and Expressing interactions grew much faster than Doing activities over the study period. In July 2024, usage was evenly split between Asking and Doing, but by June 2025 the distribution had shifted to 51.6% Asking, 34.6% Doing, and 13.8% Expressing [2].

### Work-Related Activity Patterns

When examining work-related usage specifically, the research finds that about 81% of work-related messages are associated with two broad categories of work activities: 1) obtaining, documenting, and interpreting information; and 2) making decisions, giving advice, solving problems, and thinking creatively [2].

Remarkably, work activities associated with ChatGPT usage are highly similar across very different kinds of occupations [2]. The activities of Getting Information and Making Decisions and Solving Problems appear in the top five message frequencies across nearly all occupations, from management and business to STEM to administrative and sales roles [2].

### User Satisfaction and Quality Patterns

User satisfaction remains consistently high throughout the study period, with positive interactions outnumbering negative ones by approximately 4:1 [1][3]. Asking messages consistently receive higher quality ratings compared to other interaction types, both from automated satisfaction classifiers and direct user feedback [2].

### Economic Impact and Implications

The authors conclude that ChatGPT's strongest economic value lies in serving as a decision-support tool that helps users make choices, think through problems, and produce better writing [1]. This distinguishes it from traditional search engines by generating tailored, actionable outputs rather than simply providing links to existing information.

ChatGPT provides economic value primarily through decision support, which is especially important in knowledge-intensive jobs where better decision-making directly increases productivity [1][2]. Information-seeking and decision support emerge as the most common ChatGPT use cases across most job categories [2].

The research suggests that while most economic analysis of AI has focused on productivity impacts in paid work, the impact on activity outside of work (home production) may be on a similar scale and possibly larger [1]. This finding aligns with Collis and Brynjolfsson (2025), who estimated a consumer surplus of at least $97 billion in 2024 alone in the US from generative AI.

The benefits appear to be concentrated among users with higher levels of education and better jobs, suggesting that while demographic gaps in basic usage are closing, disparities in deriving economic value may persist [4].

### Broader Societal Integration Patterns

The research reveals that ChatGPT is evolving from being perceived as primarily a "work tool" to becoming a companion for decision-making, creativity, and everyday problem-solving [3]. The platform's biggest value may lie in everyday decision support, learning, and creativity, fundamentally reshaping how people think rather than just how they work [3].

Importantly, the data suggests that people do not primarily turn to AI for social connection, but rather for cognitive leverage [4]. This finding has significant implications for understanding the role of AI in society and its potential impacts on human relationships and social structures.

The study represents one of the first comprehensive analyses of LLM chatbot usage at scale [1], providing crucial empirical grounding for policy discussions and economic analysis of AI's societal impact. The findings suggest that AI chatbots are becoming integrated into daily life in ways that extend far beyond workplace productivity, with implications for education, decision-making, and personal development that deserve further investigation.

### Sources

[1] How People Use ChatGPT | NBER: https://www.nber.org/papers/w34255
[2] [PDF] How People Use ChatGPT - National Bureau of Economic Research: https://www.nber.org/system/files/working_papers/w34255/w34255.pdf
[3] How People Are Really Using ChatGPT - - Mike Jeffs: https://mikejeffs.com/blog/how-people-are-really-using-chatgpt/
[4] Harvard's Analysis of 1.5M ChatGPT Chats Reveals Surprising User ...: https://evakeiffenheim.substack.com/p/harvards-analysis-of-15m-chatgpt


Research workflow completed!


## Understanding the Output

Let's break down what happened:

### Phase 1: Clarification
The system checked if your request was clear. Since you provided a PDF and specific questions, it likely proceeded without clarification.

### Phase 2: Research Brief
Your request was transformed into a detailed research brief that guides the supervisor's delegation strategy.

### Phase 3: Supervisor Delegation
The supervisor analyzed the brief and decided how to break down the research:
- Used `think_tool` to plan strategy
- Called `ConductResearch` multiple times to delegate to parallel researchers
- Each delegation specified a focused research topic

### Phase 4: Parallel Research
Multiple researchers worked simultaneously:
- Each researcher used web search tools to gather information
- Used `think_tool` to reflect after each search
- Decided when they had enough information
- Compressed their findings into clean summaries

### Phase 5: Final Report
All research findings were synthesized into a comprehensive report with:
- Well-structured sections
- Inline citations
- Sources listed at the end
- Balanced coverage of all findings

#### 🏗️ Activity #1: Try Different Configurations

You can experiment with different settings to see how they affect the research.  You may select three or more of the following settings (or invent your own experiments) and describe the results.

### Increase Parallelism
```python
"max_concurrent_research_units": 10  # More researchers working simultaneously
```

### Deeper Research
```python
"max_researcher_iterations": 8   # Supervisor can delegate more times
"max_react_tool_calls": 15      # Each researcher can search more
```

### Use Anthropic Native Search
```python
"search_api": "anthropic"  # Use Claude's built-in web search
```

### Disable Clarification
```python
"allow_clarification": False  # Skip clarification phase
```

## Key Takeaways

### Architecture Benefits
1. **Dynamic Decomposition** - Research structure emerges from the question, not predefined
2. **Parallel Efficiency** - Multiple researchers work simultaneously
3. **ReAct Quality** - Strategic reflection improves search decisions
4. **Scalability** - Handles token limits gracefully through compression
5. **Flexibility** - Easy to add new tools and capabilities

### When to Use This Pattern
- **Complex research questions** that need multi-angle investigation
- **Comparison tasks** where parallel research on different topics is beneficial
- **Open-ended exploration** where structure should emerge dynamically
- **Time-sensitive research** where parallel execution speeds up results

### When to Use Section-Based Instead
- **Highly structured reports** with predefined format requirements
- **Template-based content** where sections are always the same
- **Sequential dependencies** where later sections depend on earlier ones
- **Budget constraints** where token efficiency is critical

## Next Steps

### Extend the System
1. **Add MCP Tools** - Integrate specialized tools for your domain
2. **Custom Prompts** - Modify prompts for specific research types
3. **Different Models** - Try different Claude versions or mix models
4. **Persistence** - Use a real database for checkpointing instead of memory

### Learn More
- [LangGraph Documentation](https://langchain-ai.github.io/langgraph/)
- [Open Deep Research Repo](https://github.com/langchain-ai/open_deep_research)
- [Anthropic Claude Documentation](https://docs.anthropic.com/)
- [Tavily Search API](https://tavily.com/)

### Deploy
- Use LangGraph Cloud for production deployment
- Add proper error handling and logging
- Implement rate limiting and cost controls
- Monitor research quality and costs